# 05 - Test Report Agent (Gemma 4) di Google Colab

Notebook ini **khusus untuk dijalankan di Google Colab dengan GPU** — beda
dari notebook lain di folder ini yang jalan di CPU biasa. Tujuannya
mencoba `utils/report_agent.py::generate_report()` (narasi "Alasan" dari
LLM Gemma 4) secara langsung, **tanpa lewat UI Streamlit** — cara paling
cepat untuk lihat kualitas narasinya & berapa lama generate-nya sebelum
dites lewat halaman "Pengajuan Credit Baru" yang sesungguhnya.

> ⚠️ **Notebook ini belum pernah dieksekusi** — dibuat & ditulis di
> environment tanpa GPU (tidak bisa download/jalankan Gemma 4 di sana),
> jadi tidak ada output tersimpan di sel manapun. Jalankan dari atas ke
> bawah di Colab.

## Sebelum mulai

1. **Ganti runtime ke GPU**: menu `Runtime` → `Change runtime type` →
   pilih `T4 GPU` (gratis) atau lebih tinggi kalau tersedia.
2. **Akses model Gemma 4**: buka
   [huggingface.co/google/gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it),
   login, dan setujui lisensi model (biasanya perlu klik "Agree and access
   repository" sekali). Siapkan HuggingFace access token
   (Settings → Access Tokens) untuk login di Sel 3 di bawah.
3. Pastikan perubahan terbaru project sudah ke-push ke GitHub (`git push`)
   sebelum menjalankan notebook ini — sel clone di bawah mengambil dari
   `origin/main`.

In [ ]:
!nvidia-smi

## 1. Clone Repo & Install Dependencies

`torch`/`transformers`/`accelerate` sengaja **tidak** ada di
`requirements.txt` project (itu khusus untuk deploy Streamlit Cloud yang
tidak punya GPU) — jadi diinstall terpisah di sini.

In [ ]:
REPO_URL = "https://github.com/indahsyafhyra12/Capstone-Project-ODP-DA-Asek.git"
REPO_DIR = "Capstone-Project-ODP-DA-Asek"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U transformers accelerate huggingface_hub

## 2. Login HuggingFace (wajib untuk model gated seperti Gemma)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Import Modul Project

Dijalankan dari root repo (`%cd` di atas), jadi `utils.*` bisa langsung
di-import tanpa perlu `sys.path` tambahan.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import pandas as pd

from utils.feature_builder import load_raw_tables
from utils.risk_ml_pipeline import predict_credit_screening
from utils.report_agent import generate_report, generate_reports_batch, MODEL_ID

print(f"Model target: {MODEL_ID}")

master_dataset = pd.read_csv("data/processed/master_dataset.csv", dtype={"NIK": str})
master_scored = pd.read_csv("data/processed/master_scored.csv", dtype={"NIK": str})
print(f"master_dataset: {master_dataset.shape}, master_scored: {master_scored.shape}")

## 4. Preload Model (Sekali Saja)

Panggilan pertama ke `generate_report()` yang men-trigger download +
load model ke GPU (bisa beberapa menit tergantung koneksi) — dijalankan
di sel terpisah supaya waktu download tidak tercampur ke timing generate
per-nasabah di sel berikutnya.

In [ ]:
_warm_start = time.perf_counter()
from utils.report_agent import _load_model
_processor, _model = _load_model()
print(f"Model dimuat dalam {time.perf_counter() - _warm_start:.1f}s")
print(f"Device: {_model.device}")

## 5. Test 5 Kasus Representatif (sama seperti notebook 04)

Mencakup: Layak, Layak Bersyarat, Perlu Review Ulang, Tidak Layak (lewat
ML), dan Tidak Layak (hard-reject DHN — STAGE 1, tidak pernah sampai ML/
LLM sama sekali, buat lihat narasinya tetap masuk akal utk kasus ini).

In [ ]:
TEST_IDS = {
    "APP202600001": "Layak",
    "APP202600008": "Layak Bersyarat",
    "APP202600003": "Perlu Review Ulang",
    "APP202600576": "Tidak Layak (skor rendah, lewat ML)",
    "APP202600020": "Tidak Layak (hard-reject DHN, STAGE 1)",
}

results = []
for app_id, label in TEST_IDS.items():
    row = master_dataset[master_dataset["application_id"] == app_id].iloc[0]
    company_name = row["company_name"]
    pred = predict_credit_screening(row.to_dict())
    prompt_row = {"company_name": company_name, **pred}

    start = time.perf_counter()
    narrative = generate_report(prompt_row)
    elapsed = time.perf_counter() - start

    is_fallback = narrative == pred["insight"]
    results.append({
        "application_id": app_id, "expected": label, "company_name": company_name,
        "decision": pred["decision"], "risk_score": pred["risk_score"],
        "elapsed_sec": round(elapsed, 2), "is_fallback": is_fallback,
        "insight_rule_based": pred["insight"], "narrative_llm": narrative,
    })

    print("=" * 80)
    print(f"{app_id} ({company_name}) — ekspektasi: {label}")
    print(f"Decision: {pred['decision']} | risk_score: {pred['risk_score']} | waktu generate: {elapsed:.2f}s")
    print(f"Fallback ke rule-based? {'YA - cek log warning di atas' if is_fallback else 'TIDAK - narasi LLM asli'}")
    print()
    print("[Insight rule-based]")
    print(pred["insight"])
    print()
    print("[Narasi LLM]")
    print(narrative)
print("=" * 80)

In [ ]:
summary_df = pd.DataFrame(results)[["application_id", "expected", "decision", "risk_score", "elapsed_sec", "is_fallback"]]
summary_df

## 6. Cek Konsistensi Guardrail Manual

Baca ulang tiap narasi di atas dan cek manual: apakah kata-katanya tidak
bertentangan dengan `decision`-nya (mis. narasi utk "Tidak Layak" tidak
menyiratkan disetujui, dan sebaliknya)? Kolom `is_fallback=True` di tabel
atas berarti guardrail otomatis (`_sanity_check`) sudah mendeteksi masalah
duluan dan fallback ke rule-based — kalau itu terjadi cukup sering, cek log
`WARNING` di output Sel 6 untuk lihat narasi asli yang ditolak.

## 7. Test Batch (`generate_reports_batch`)

Sample kecil (10 baris, hanya yang lolos hard-rule / `risk_score` terisi)
dari `master_scored.csv` — simulasi pemakaian notebook untuk mengisi ulang
kolom insight versi LLM secara massal.

In [ ]:
sample = master_scored[master_scored["risk_score"].notna()].sample(10, random_state=42)

batch_start = time.perf_counter()
sample_narratives = generate_reports_batch(sample)
batch_elapsed = time.perf_counter() - batch_start

print(f"Total waktu utk {len(sample)} baris: {batch_elapsed:.1f}s (rata-rata {batch_elapsed/len(sample):.2f}s/baris)")

batch_preview = sample[["application_id", "company_name", "decision", "risk_score"]].copy()
batch_preview["narrative_llm"] = sample_narratives.values
batch_preview

## Ringkasan & Langkah Selanjutnya

- Kalau narasi di Bagian 5 sudah bagus & `is_fallback` jarang `True`,
  `generate_report()` siap dipakai — tidak perlu ubah apa pun di
  `utils/report_agent.py`.
- Kalau `is_fallback` sering `True`, buka log `WARNING` di atas untuk
  lihat narasi asli yang ditolak guardrail — mungkin perlu sesuaikan
  `_sanity_check()` (terlalu ketat) atau `SYSTEM_PROMPT` (model sering
  keluar dari format).
- Waktu generate per-nasabah di kolom `elapsed_sec` (Bagian 5) menentukan
  perlu tidaknya UX tambahan (progress indicator, dsb) di halaman
  "Pengajuan Credit Baru" — saat ini sudah ada `st.spinner`, cek apakah
  itu cukup atau perlu progress bar kalau ternyata generate-nya lama.
- Untuk test UI Streamlit-nya juga (bukan cuma fungsi `generate_report()`
  langsung), jalankan `streamlit run app.py` di Colab lalu expose lewat
  `pyngrok`/`localtunnel` — beri tahu kalau mau notebook terpisah utk itu.